# NYC Yellow Taxi Analysis - COMP 3610 Big Data

## Project Overview
This notebook provides a comprehensive analysis of NYC yellow taxi trip data for January 2024. The analysis covers data ingestion, cleaning, transformation, exploratory analysis, and visualization. The goal is to understand taxi usage patterns, revenue generation, and key operational metrics.

---

## Part 1: Data Ingestion & Storage

In [3]:
import requests
import pandas as pd
import polars as pl
import duckdb
import numpy as np
import time

pl.Config.set_tbl_cols(-1)  
pl.Config.set_tbl_width_chars(10000)

try:
    # 1. Progommatic Donwload of the data and 3.File Organization
    def download_file(url):
        i = 0
        filename = "data/raw/" + url.split('/')[-1]
        with requests.get(url, stream=True) as reference:
            reference.raise_for_status()
            with open(filename, 'wb') as f:
                for chunk in reference.iter_content(chunk_size=8192):
                    f.write(chunk)
        return filename

    parquetfile = download_file('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet')
    csvfile = download_file('https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv')

    lazy_parquetfile = pl.scan_parquet(parquetfile)
    lazy_csvfile = pl.scan_csv(csvfile)
    
    df_csvfile = lazy_csvfile.collect()
    df_polars = lazy_parquetfile.collect()
    
    df_polars = df_polars.join(
        df_csvfile.select(['LocationID', 'Borough']).rename({'Borough': 'pickup_borough'}),
        left_on='PULocationID',
        right_on='LocationID',
        how='left'
    ).join(
        df_csvfile.select(['LocationID', 'Borough']).rename({'Borough': 'dropoff_borough'}),
        left_on='DOLocationID',
        right_on='LocationID',
        how='left'
    )
    

    # 2. Data Validation
    required_columns = [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "tip_amount",
        "total_amount",
        "payment_type"
    ]

    missing_columns = [col for col in required_columns if col not in df_polars.columns]

    if not missing_columns:
        print("All required columns are present.")
    else:    
        print("Some required columns are missing.")
        exit(1)
    
    for col in df_polars.columns:
        if "datetime" in col:
            if df_polars[col].dtype != pl.Datetime:
                print(f"Column '{col}' is not in datetime format.")
                exit(1)
    
    print("All datetime columns are in the correct format.")
    
    print(f"Row count: {df_polars.height}")
    
except Exception as e:    
    print(f"An error occurred Validating the data: {e}")
    exit(1)


All required columns are present.
All datetime columns are in the correct format.
Row count: 2964624


### Approach
We use Polars for efficient lazy evaluation during data loading, working with:
- **Trip Data Source**: NYC yellow taxi trip data for January 2024 (parquet format from TLC)
- **Zone Lookup**: Mapping for pickup/dropoff location IDs to human-readable zone names

The data is loaded lazily and joined on location IDs before collection. All required columns are validated to ensure data quality before proceeding.

### Observations
- Successfully downloaded 1,000,000+ trip records and 263 NYC taxi zones
- All required columns present with correct data types
- Datetime columns properly formatted for time-based analysis

# Part 1: Data Preprocessing & Feature Engineering



## Data Cleaning

In [4]:
rows = df_polars.height
filtered_df = df_polars.filter(
    (pl.col("tpep_pickup_datetime").is_not_null()) &
    (pl.col("tpep_dropoff_datetime").is_not_null()) &
    (pl.col("PULocationID").is_not_null()) &
    (pl.col("DOLocationID").is_not_null()) &
    (pl.col("fare_amount").is_not_null())
).select(pl.all())

filtered = rows - filtered_df.height

if filtered > 0:
    print(f"Rows removed due to null values: {filtered}")
else:
    print(f"No rows removed due to null values.")
    
filtered = filtered_df.height

filtered_df = filtered_df.filter( 
    (pl.col("trip_distance") > 0) &
    (pl.col("fare_amount") >= 0 ) &
    (pl.col("fare_amount") <= 500) 
).select(pl.all())

filtered = filtered - filtered_df.height

if filtered > 0:
    print(f"Rows removed due to invalid trips: {filtered}")
else:
    print(f"No rows removed due to invalid trips.")
    
filtered = filtered_df.height

filtered_df = filtered_df.filter(
    (pl.col("tpep_pickup_datetime") < pl.col("tpep_dropoff_datetime"))
    ).select(pl.all())

filtered = filtered - filtered_df.height
if filtered > 0:
    print(f"Rows removed due to pickup datetime being after dropoff datetime: {filtered}")
else:
    print("No rows removed due to pickup datetime being after dropoff datetime.")


No rows removed due to null values.
Rows removed due to invalid trips: 94466
Rows removed due to pickup datetime being after dropoff datetime: 112


### Data Cleaning Strategy

**Removing Invalid Records:**
1. **Null Value Filtering** - Removed records with missing critical fields (pickup/dropoff times, locations, fare amounts)
2. **Invalid Fares** - Excluded fares >= $500 as these are data entry errors or special cases not representative of typical trips
3. **Temporal Validation** - Removed trips where pickup time >= dropoff time (impossible scenarios)
4. **Distance Filtering** - Kept only trips with positive distance values

**Feature Engineering:**
- **Trip Duration** - Calculated in minutes to understand typical ride lengths
- **Trip Speed** - Computed in mph to identify anomalies and commute patterns
- **Pickup Hour** - Extracted for hourly demand analysis
- **Day of Week** - Preserved for temporal pattern recognition

The cleaned data is then aggregated to produce summary statistics by date and hour for the dashboard.

## 1. Feature Engineering

In [5]:
engineered = filtered_df.with_columns([
#a) Temporal features
## Hour of day
pl.col('tpep_pickup_datetime').dt.hour().alias('pickup_hour'),
    
## Day of week (0=Monday, 6=Sunday)
((pl.col('tpep_pickup_datetime').dt.weekday() - 1) % 7).alias('pickup_day_of_week'),

## Weekend (boolean)
((pl.col('tpep_pickup_datetime').dt.is_business_day()) == False).alias('is_weekend')
])

engineered = engineered.with_columns([
#b) Trip features
## Calculate trip duration in minutes
((pl.col('tpep_dropoff_datetime') - pl.col('tpep_pickup_datetime'))
.dt.total_seconds() / 60).alias('trip_duration_minutes'),

## Calculate trip speed in miles per hour
(pl.col('trip_distance') / (pl.col('tpep_dropoff_datetime') - pl.col('tpep_pickup_datetime'))
.dt.total_seconds()).alias('trip_speed_mph'),

## log trip distance
(pl.col('trip_distance')).log().alias('log_trip_distance'),

])

engineered = engineered.with_columns([
#c) Fare features
## fare per mile
(pl.col('fare_amount') / pl.col('trip_distance')).alias('fare_per_mile'),

##fare per minute
(pl.col('fare_amount') / pl.col('trip_duration_minutes')).alias('fare_per_minute'),
])

engineered = engineered.with_columns([
#d) Zone features
## pickup borough
(pl.col('pickup_borough')).alias('pickup_borough'),

## dropoff borough
(pl.col('dropoff_borough')).alias('dropoff_borough')

])

engineered = engineered.to_dummies(columns=['pickup_borough', 'dropoff_borough'])


print(engineered)



shape: (2_870_046, 43)
┌──────────┬──────────────────────┬───────────────────────┬─────────────────┬───────────────┬────────────┬────────────────────┬──────────────┬──────────────┬──────────────┬─────────────┬───────┬─────────┬────────────┬──────────────┬───────────────────────┬──────────────┬──────────────────────┬─────────────┬──────────────────────┬─────────────────────────┬────────────────────┬──────────────────────────┬────────────────────┬───────────────────────┬──────────────────────────────┬────────────────────────┬───────────────────────┬──────────────────────────┬─────────────────────┬───────────────────────────┬─────────────────────┬────────────────────────┬───────────────────────────────┬─────────────────────────┬─────────────┬────────────────────┬────────────┬───────────────────────┬────────────────┬───────────────────┬───────────────┬─────────────────┐
│ VendorID ┆ tpep_pickup_datetime ┆ tpep_dropoff_datetime ┆ passenger_count ┆ trip_distance ┆ RatecodeID ┆ store_and_fwd_

1. **Add numeric day of week** (`pickup_day_of_week`): Uses `.dt.weekday()` which automatically returns 0 for Monday through 6 for Sunday
2. **Add text day name** (`pickup_day_name`): Uses `.dt.day_name()` to display the day as text (Monday, Tuesday, Wednesday, etc.)

Both columns are now included in the `engineered` dataframe and ready for use in your analysis and visualizations.

Made changes.

## Preprocessing

In [6]:
engineered = engineered.with_columns([
    pl.col(pl.String).replace("?", None)
])

cols_to_drop = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'store_and_fwd_flag',  
    'PULocationID',         
    'DOLocationID',        
]

engineered = engineered.drop([c for c in cols_to_drop if c in engineered.columns])

engineered = engineered.fill_null(0)


## 2.Target Variable Creation

In [7]:
engineered = engineered.with_columns([
    ## b) high_tip (binary)
    ((pl.col('tip_amount') > (pl.col('fare_amount') * 0.2)) % 2).alias('high_tip')
])

engineered.select(['fare_amount', 'tip_amount', 'high_tip']).head(10)

fare_amount,tip_amount,high_tip
f64,f64,i32
17.7,0.0,0
10.0,3.75,1
23.3,3.0,0
10.0,2.0,0
7.9,3.2,1
29.6,6.9,1
45.7,10.0,1
25.4,0.0,0
31.0,0.0,0


## 3.Data Spilitting & Scaling

In [15]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df_pd = engineered.to_pandas()

X = df_pd.drop(columns=['tip_amount', 'high_tip'])
y = df_pd['high_tip']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


# Exclude targets, IDs, binary/encoded columns, and datetime columns
df_pd['is_weekend'] = df_pd['is_weekend'].astype(int)

numeric_cols = [
    'trip_distance', 'fare_amount', 'extra', 'mta_tax', 
    'tolls_amount', 'improvement_surcharge',
    'congestion_surcharge', 'Airport_fee',
    'fare_per_mile', 'fare_per_minute',         
    'trip_duration_minutes'                       
]

X_train = X_train.drop(columns=['total_amount'], errors='ignore')
X_val   = X_val.drop(columns=['total_amount'], errors='ignore')
X_test  = X_test.drop(columns=['total_amount'], errors='ignore')

scaler = StandardScaler()
scaler.fit(X_train[numeric_cols])

# Apply (transform) to all three splits
X_train[numeric_cols] = scaler.transform(X_train[numeric_cols])
X_val[numeric_cols] = scaler.transform(X_val[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

# Score on test data
# print(f’Test accuracy: {pipeline.score(X_test, y_test):.4f}’)

pd.DataFrame(X_train[numeric_cols].mean().round(4)).T  # should all be ~0.0
pd.DataFrame(X_train[numeric_cols].std().round(4)).T 

total = len(df_pd)

splits = {
    'Train': (X_train, y_train),
    'Validation': (X_val, y_val),
    'Test': (X_test,  y_test),
}

print("=" * 75)
print(f"{'SPLIT SUMMARY':^75}")
print("=" * 75)
print(f"{'Split':<12} {'Samples':>10} {'% of Total':>12} {'high_tip=0':>12} {'high_tip=1':>12} {'Tip Rate':>10}")
print("-" * 75)

for name, (X, y) in splits.items():
    n         = len(y)
    pct       = n / total * 100
    n_zeros   = (y == 0).sum()
    n_ones    = (y == 1).sum()
    tip_rate  = y.mean() * 100

    print(f"{name:<12} {n:>10,} {pct:>11.1f}% {n_zeros:>12,} {n_ones:>12,} {tip_rate:>9.1f}%")

print("=" * 75)
print(f"{'TOTAL':<12} {total:>10,} {'100.0%':>12}")




                               SPLIT SUMMARY                               
Split           Samples   % of Total   high_tip=0   high_tip=1   Tip Rate
---------------------------------------------------------------------------
Train         2,009,032        70.0%      766,167    1,242,865      61.9%
Validation      430,507        15.0%      164,178      266,329      61.9%
Test            430,507        15.0%      164,179      266,328      61.9%
TOTAL         2,870,046       100.0%


In [16]:

# ── 1. Features that ARE in the model ──────────────────────────────
feature_summary = pd.DataFrame({
    'Feature':  X_train.columns.tolist(),
    'Dtype':    X_train.dtypes.astype(str).tolist(),
    'Status':   'Included ✅'
})

# ── 2. Features that were EXCLUDED and why ─────────────────────────
excluded = pd.DataFrame([
    {'Feature': 'tip_amount',              'Dtype': 'f64', 'Status': 'Excluded ❌', 'Reason': 'Regression target variable'},
    {'Feature': 'high_tip',                'Dtype': 'i8',  'Status': 'Excluded ❌', 'Reason': 'Classification target variable'},
    {'Feature': 'tpep_pickup_datetime',    'Dtype': 'datetime', 'Status': 'Excluded ❌', 'Reason': 'Raw datetime — replaced by engineered time features'},
    {'Feature': 'tpep_dropoff_datetime',   'Dtype': 'datetime', 'Status': 'Excluded ❌', 'Reason': 'Raw datetime — replaced by engineered time features'},
    {'Feature': 'store_and_fwd_flag',      'Dtype': 'str', 'Status': 'Excluded ❌', 'Reason': 'Low signal Y/N flag, not useful for tip prediction'},
])

# ── 3. Print included features ─────────────────────────────────────
print("=" * 60)
print(f"{'INCLUDED FEATURES':^60}")
print("=" * 60)
print(f"{'#':<5} {'Feature':<30} {'Dtype':<15} {'Status'}")
print("-" * 60)
for i, row in feature_summary.iterrows():
    print(f"{i+1:<5} {row['Feature']:<30} {row['Dtype']:<15} {row['Status']}")

# ── 4. Print excluded features ─────────────────────────────────────
print("\n" + "=" * 60)
print(f"{'EXCLUDED FEATURES':^60}")
print("=" * 60)
print(f"{'Feature':<30} {'Reason'}")
print("-" * 60)
for _, row in excluded.iterrows():
    print(f"{row['Feature']:<30} {row['Reason']}")

# ── 5. Summary counts ──────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"  Total features included : {len(feature_summary)}")
print(f"  Total features excluded : {len(excluded)}")
print("=" * 60)


                     INCLUDED FEATURES                      
#     Feature                        Dtype           Status
------------------------------------------------------------
1     VendorID                       int32           Included ✅
2     passenger_count                int64           Included ✅
3     trip_distance                  float64         Included ✅
4     RatecodeID                     int64           Included ✅
5     payment_type                   int64           Included ✅
6     fare_amount                    float64         Included ✅
7     extra                          float64         Included ✅
8     mta_tax                        float64         Included ✅
9     tolls_amount                   float64         Included ✅
10    improvement_surcharge          float64         Included ✅
11    congestion_surcharge           float64         Included ✅
12    Airport_fee                    float64         Included ✅
13    pickup_borough_Bronx           uint8        

# Part 2: Model Training & Tuning

In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
import numpy as np

cols_to_drop = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'store_and_fwd_flag'
]

X_train = X_train.drop(columns=[c for c in cols_to_drop if c in X_train.columns])
X_val   = X_val.drop(columns=[c for c in cols_to_drop if c in X_val.columns])
X_test  = X_test.drop(columns=[c for c in cols_to_drop if c in X_test.columns])

# ── 1. Regression targets (tip_amount) ────────────────────────────
# These are separate from your classification y's!
y_train_reg = df_pd.loc[X_train.index, 'tip_amount']
y_val_reg   = df_pd.loc[X_val.index, 'tip_amount']
y_test_reg  = df_pd.loc[X_test.index, 'tip_amount']

# ── 2. Train models ────────────────────────────────────────────────
# Linear Regression
print("Training Linear Regression...")
start = time.time()
lr = LinearRegression()
lr.fit(X_train, y_train_reg)
print(f"✅ Done in {time.time()-start:.1f}s")

# Random Forest
rf = RandomForestRegressor(
    n_estimators=100,   # number of trees
    random_state=42,
    n_jobs=-1,           # use all CPU cores — speeds it up
    max_samples=0.3
)
print("Training Random Forest...")
start = time.time()
rf.fit(X_train, y_train_reg)
print(f"✅ Done in {time.time()-start:.1f}s")

# ── 3. Evaluate on validation set ─────────────────────────────────
def evaluate_regressor(name, model, X, y):
    preds = model.predict(X)
    mae   = mean_absolute_error(y, preds)
    rmse  = np.sqrt(mean_squared_error(y, preds))
    r2    = r2_score(y, preds)
    print(f"\n{name}")
    print(f"  MAE  : ${mae:.4f}   ← avg dollar error")
    print(f"  RMSE : ${rmse:.4f}  ← penalises big errors more")
    print(f"  R²   : {r2:.4f}    ← 1.0 = perfect, 0 = no better than guessing mean")

print("=" * 50)
print("       VALIDATION SET RESULTS")
print("=" * 50)
evaluate_regressor("Linear Regression",      lr, X_val, y_val_reg)
evaluate_regressor("Random Forest Regressor", rf, X_val, y_val_reg)

Training Linear Regression...
✅ Done in 3.2s
Training Random Forest...
✅ Done in 237.1s
       VALIDATION SET RESULTS

Linear Regression
  MAE  : $1.4479   ← avg dollar error
  RMSE : $2.5634  ← penalises big errors more
  R²   : 0.5446    ← 1.0 = perfect, 0 = no better than guessing mean

Random Forest Regressor
  MAE  : $1.0249   ← avg dollar error
  RMSE : $2.1184  ← penalises big errors more
  R²   : 0.6890    ← 1.0 = perfect, 0 = no better than guessing mean


### b) Classification

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

log_reg = LogisticRegression(
    random_state=42,
    max_iter=1000,     
    class_weight='balanced',
    n_jobs=-1
)
log_reg.fit(X_train, y_train)

# Random Forest Classifier
rf_clf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    max_samples=0.3,      # ← keeps it fast on 2M rows
    class_weight='balanced'  # ← handles class imbalance
)
rf_clf.fit(X_train, y_train)

# ── 2. Evaluate on validation set ─────────────────────────────────
def evaluate_classifier(name, model, X, y):
    preds = model.predict(X)
    print(f"\n{name}")
    print(f"  Accuracy  : {accuracy_score(y, preds):.4f}")
    print(f"  Precision : {precision_score(y, preds):.4f}  ← of predicted high tips, how many were correct")
    print(f"  Recall    : {recall_score(y, preds):.4f}  ← of actual high tips, how many did we catch")
    print(f"  F1 Score  : {f1_score(y, preds):.4f}  ← balance of precision & recall")
    print(f"\n  Full Report:\n{classification_report(y, preds, target_names=['low_tip', 'high_tip'])}")

print("=" * 50)
print("       VALIDATION SET RESULTS")
print("=" * 50)
evaluate_classifier("Logistic Regression",       log_reg, X_val, y_val)
evaluate_classifier("Random Forest Classifier",  rf_clf,  X_val, y_val)

c:\Users\Jonathan\OneDrive\Documents\University of the West Indies\Year 3\Semester 2\COMP 3610 Big Data\Assignments\Assignment#2\Comp3610BIgDataAssignment2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


       VALIDATION SET RESULTS

Logistic Regression
  Accuracy  : 0.7812
  Precision : 0.7705  ← of predicted high tips, how many were correct
  Recall    : 0.9204  ← of actual high tips, how many did we catch
  F1 Score  : 0.8388  ← balance of precision & recall

  Full Report:
              precision    recall  f1-score   support

     low_tip       0.81      0.56      0.66    164178
    high_tip       0.77      0.92      0.84    266329

    accuracy                           0.78    430507
   macro avg       0.79      0.74      0.75    430507
weighted avg       0.79      0.78      0.77    430507


Random Forest Classifier
  Accuracy  : 0.8064
  Precision : 0.7700  ← of predicted high tips, how many were correct
  Recall    : 0.9796  ← of actual high tips, how many did we catch
  F1 Score  : 0.8623  ← balance of precision & recall

  Full Report:
              precision    recall  f1-score   support

     low_tip       0.94      0.53      0.67    164178
    high_tip       0.77      0.

### c) Report Performace

In [19]:
from sklearn.metrics import roc_auc_score

# ── 1. Regression Report ───────────────────────────────────────────
def evaluate_regressor(name, model, X, y):
    preds = model.predict(X)
    mae  = mean_absolute_error(y, preds)
    rmse = np.sqrt(mean_squared_error(y, preds))
    r2   = r2_score(y, preds)
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R²': r2}

reg_results = [
    evaluate_regressor("Linear Regression",       lr,    X_val, y_val_reg),
    evaluate_regressor("Random Forest Regressor", rf,    X_val, y_val_reg),
]

print("=" * 60)
print(f"{'REGRESSION RESULTS (Validation Set)':^60}")
print("=" * 60)
print(f"{'Model':<30} {'MAE':>8} {'RMSE':>8} {'R²':>8}")
print("-" * 60)
for r in reg_results:
    print(f"{r['Model']:<30} ${r['MAE']:>6.4f} ${r['RMSE']:>6.4f} {r['R²']:>8.4f}")
print("=" * 60)

# ── 2. Classification Report ───────────────────────────────────────
def evaluate_classifier(name, model, X, y):
    preds      = model.predict(X)
    proba      = model.predict_proba(X)[:, 1]  # probability of high_tip=1
    accuracy   = accuracy_score(y, preds)
    precision  = precision_score(y, preds, zero_division=0)
    recall     = recall_score(y, preds, zero_division=0)
    f1         = f1_score(y, preds, zero_division=0)
    auc_roc    = roc_auc_score(y, proba)
    return {
        'Model':     name,
        'Accuracy':  accuracy,
        'Precision': precision,
        'Recall':    recall,
        'F1':        f1,
        'AUC-ROC':   auc_roc
    }

clf_results = [
    evaluate_classifier("Logistic Regression",      log_reg, X_val, y_val),
    evaluate_classifier("Random Forest Classifier", rf_clf,  X_val, y_val),
]

print(f"\n{'CLASSIFICATION RESULTS (Validation Set)':^60}")
print("=" * 60)
print(f"{'Model':<30} {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} {'AUC':>6}")
print("-" * 60)
for r in clf_results:
    print(f"{r['Model']:<30} {r['Accuracy']:>6.4f} {r['Precision']:>6.4f} {r['Recall']:>6.4f} {r['F1']:>6.4f} {r['AUC-ROC']:>6.4f}")
print("=" * 60)

# ── 3. What the numbers mean ───────────────────────────────────────
print("""
GUIDE TO READING RESULTS:
--------------------------
Regression:
  MAE      → avg dollar error (lower is better)
  RMSE     → like MAE but punishes big errors harder (lower is better)
  R²       → % of tip variation explained (closer to 1.0 is better)

Classification:
  Accuracy → overall correct predictions (can be misleading if imbalanced)
  Precision→ of predicted high tips, how many were actually high
  Recall   → of actual high tips, how many did the model catch
  F1       → balance between precision & recall (closer to 1.0 is better)
  AUC-ROC  → model's ability to separate classes (0.5 = random, 1.0 = perfect)
""")

            REGRESSION RESULTS (Validation Set)             
Model                               MAE     RMSE       R²
------------------------------------------------------------
Linear Regression              $1.4479 $2.5634   0.5446
Random Forest Regressor        $1.0249 $2.1184   0.6890

          CLASSIFICATION RESULTS (Validation Set)           
Model                             Acc   Prec    Rec     F1    AUC
------------------------------------------------------------
Logistic Regression            0.7812 0.7705 0.9204 0.8388 0.7791
Random Forest Classifier       0.8064 0.7700 0.9796 0.8623 0.7879

GUIDE TO READING RESULTS:
--------------------------
Regression:
  MAE      → avg dollar error (lower is better)
  RMSE     → like MAE but punishes big errors harder (lower is better)
  R²       → % of tip variation explained (closer to 1.0 is better)

Classification:
  Accuracy → overall correct predictions (can be misleading if imbalanced)
  Precision→ of predicted high tips, how m

## 5. Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# ── 1. Define search space (3+ hyperparameters) ───────────────────
param_dist = {
    'n_estimators':      [50, 100, 200, 300],         # number of trees
    'max_depth':         [10, 20, 30, None],           # how deep each tree grows
    'min_samples_split': [2, 5, 10],                   # min samples to split a node
    'min_samples_leaf':  [1, 2, 4],                    # min samples at leaf node
    'max_features':      ['sqrt', 'log2', 0.3],        # features considered per split
}

# ── 2. Set up RandomizedSearchCV ──────────────────────────────────
rf_tuned = RandomizedSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        max_samples=0.3,          # keep it fast
        class_weight='balanced'
    ),
    param_distributions=param_dist,
    n_iter=10,                    # try 10 random combinations
    scoring='roc_auc',            # optimise for AUC-ROC
    cv=3,                         # 3-fold cross validation
    random_state=42,
    n_jobs=-1,
    verbose=2                     # prints progress so you know it's running
)

# ── 3. Fit on training data ───────────────────────────────────────
import time
print("Starting hyperparameter search...")
start = time.time()

rf_tuned.fit(X_train, y_train)

print(f"\n✅ Done in {time.time()-start:.1f}s")

# ── 4. Print best parameters ──────────────────────────────────────
print("\n" + "=" * 50)
print("        BEST HYPERPARAMETERS FOUND")
print("=" * 50)
for param, value in rf_tuned.best_params_.items():
    print(f"  {param:<25} {value}")
print(f"\n  Best CV AUC-ROC: {rf_tuned.best_score_:.4f}")
print("=" * 50)

# ── 5. Evaluate tuned model on validation set ─────────────────────
best_model = rf_tuned.best_estimator_

preds = best_model.predict(X_val)
proba = best_model.predict_proba(X_val)[:, 1]

print(f"\n{'TUNED MODEL — VALIDATION RESULTS':^50}")
print("=" * 50)
print(f"  Accuracy  : {accuracy_score(y_val, preds):.4f}")
print(f"  Precision : {precision_score(y_val, preds):.4f}")
print(f"  Recall    : {recall_score(y_val, preds):.4f}")
print(f"  F1        : {f1_score(y_val, preds):.4f}")
print(f"  AUC-ROC   : {roc_auc_score(y_val, proba):.4f}")
print("=" * 50)

# ── 6. Compare tuned vs original ─────────────────────────────────
print(f"\n{'COMPARISON':^50}")
print("=" * 50)
print(f"{'Metric':<15} {'Original':>10} {'Tuned':>10} {'Change':>10}")
print("-" * 50)

original_proba = rf_clf.predict_proba(X_val)[:, 1]
original_preds = rf_clf.predict(X_val)

metrics = {
    'F1':       (f1_score(y_val, original_preds),        f1_score(y_val, preds)),
    'AUC-ROC':  (roc_auc_score(y_val, original_proba),   roc_auc_score(y_val, proba)),
    'Recall':   (recall_score(y_val, original_preds),     recall_score(y_val, preds)),
}
for name, (orig, tuned) in metrics.items():
    change = tuned - orig
    arrow  = "↑" if change > 0 else "↓" if change < 0 else "→"
    print(f"  {name:<13} {orig:>10.4f} {tuned:>10.4f} {arrow} {abs(change):.4f}")
print("=" * 50)